<a href="https://colab.research.google.com/github/laramalkawi81-ops/DS230-Instacart-Project/blob/main/Copy_of_09_model_interpretability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Model Interpretability

In this file, I analyze the trained model to understand how different features
affect the predictions. The goal is to interpret the model behavior rather than
improve performance.

To keep the analysis efficient, I focus on a simple and fast model that was
already trained in previous steps.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier


In [2]:

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:

DATA_PATH = "/content/drive/MyDrive/instacart_data/instacart_data"

data = pd.read_csv(f"{DATA_PATH}/model_data_sample.csv")

print("Data shape:", data.shape)
data.head()


Data shape: (1310048, 12)


,order_id,product_id,add_to_cart_order,reordered,product_freq,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,days_since_prior_order_scaled
0,2839892,7693,30,1,0.000118,137629,prior,90,5,14,4.0,-0.692348
1,2412939,47683,18,1,0.000037,111603,prior,4,4,8,7.0,-0.370070
2,1891520,40396,4,1,0.000554,87435,prior,10,4,15,8.0,-0.262644
3,1838620,15866,5,1,0.000015,68985,prior,9,1,10,3.0,-0.799774
4,2520082,14702,4,1,0.000127,177234,prior,27,1,8,14.0,0.381913




I select the same features used during model training to ensure consistency
in interpretation.


In [4]:


DATA_PATH = "/content/drive/MyDrive/instacart_data/instacart_data"

data = pd.read_csv(f"{DATA_PATH}/model_data_sample.csv")

print("Columns in the dataset:")
print(data.columns.tolist())


Columns in the dataset:
['order_id', 'product_id', 'add_to_cart_order', 'reordered', 'product_freq', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'days_since_prior_order_scaled']


In [5]:
data = pd.read_csv(f"{DATA_PATH}/orders_clean.csv")
order_products_prior = pd.read_csv(f"{DATA_PATH}/order_products_prior_clean.csv")
products = pd.read_csv(f"{DATA_PATH}/products.csv")


sample_users = data['user_id'].drop_duplicates().sample(frac=0.2, random_state=42)

sample_orders = data[data['user_id'].isin(sample_users)]
sample_prior = order_products_prior[order_products_prior['order_id'].isin(sample_orders['order_id'])]


data = sample_prior.merge(sample_orders, on='order_id', how='left')
print("Sample dataset shape:", data.shape)
data.head()

Sample dataset shape: (6550242, 12)


,order_id,product_id,add_to_cart_order,reordered,product_freq,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,days_since_prior_order_scaled
0,6,40462,1,0,0.000009,22352,prior,4,1,12,30.0,2.100730
1,6,15873,2,0,0.000002,22352,prior,4,1,12,30.0,2.100730
2,6,41897,3,0,0.000001,22352,prior,4,1,12,30.0,2.100730
3,28,35108,1,0,0.000591,98256,prior,29,3,13,6.0,-0.477496
4,28,40593,2,1,0.000215,98256,prior,29,3,13,6.0,-0.477496


In [6]:

user_features = sample_orders.groupby('user_id').agg(
    total_orders=('order_number', 'max'),
    mean_days_between_orders=('days_since_prior_order', 'mean')
).reset_index()


product_features = sample_prior.groupby('product_id').agg(
    product_reorder_rate=('reordered', 'mean')
).reset_index()


user_product_features = data.groupby(['user_id', 'product_id']).agg(
    user_product_count=('order_id', 'count')
).reset_index()


data = data.merge(user_features, on='user_id', how='left')
data = data.merge(product_features, on='product_id', how='left')
data = data.merge(user_product_features, on=['user_id','product_id'], how='left')


features = ['total_orders', 'mean_days_between_orders', 'product_reorder_rate', 'user_product_count']
target = 'reordered'

X = data[features]
y = data[target]


 Train/Test Split

We use a simple 80/20 split to simulate validation.


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (5240193, 4)
Test shape: (1310049, 4)


 Robustness Test – Gaussian Noise

We add small Gaussian noise to numeric features and check
how predictions degrade.


In [8]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

sample_idx = np.random.choice(len(X_test), size=200, replace=False)
X_test_small = X_test.iloc[sample_idx]
y_test_small = y_test.iloc[sample_idx]


rf = RandomForestRegressor(n_estimators=10, max_depth=4, random_state=42)
rf.fit(X_train, y_train)


y_pred = rf.predict(X_test_small)
rmse_baseline = np.sqrt(mean_squared_error(y_test_small, y_pred))
print("Baseline RMSE:", rmse_baseline)


X_test_noisy = X_test_small.copy()
for col in features:
    X_test_noisy[col] += np.random.normal(0, 0.01 * X_test_noisy[col].std(), size=X_test_noisy.shape[0])

y_pred_noisy = rf.predict(X_test_noisy)
rmse_noisy = np.sqrt(mean_squared_error(y_test_small, y_pred_noisy))
print("RMSE with Noise:", rmse_noisy)



Baseline RMSE: 0.30500605803998637
RMSE with Noise: 0.30500605803998637


 Robustness Test – Introduce Outliers

Randomly amplify 1% of values to simulate outliers.


In [9]:

X_test_outliers = X_test_small.copy()
n_outliers = int(0.01 * len(X_test_outliers))

for col in features:
    indices = np.random.choice(X_test_outliers.index, n_outliers, replace=False)
    X_test_outliers.loc[indices, col] *= 10  # amplify outliers

y_pred_outliers = rf.predict(X_test_outliers)
rmse_outliers = np.sqrt(mean_squared_error(y_test_small, y_pred_outliers))
print("RMSE with Outliers:", rmse_outliers)


RMSE with Outliers: 0.30522750542120225


 Robustness Test – Reduced Training Data

We train using only 50% of training data and check performance.


In [10]:

train_sample_idx = np.random.choice(len(X_train), size=int(0.3*len(X_train)), replace=False)
X_train_small = X_train.iloc[train_sample_idx]
y_train_small = y_train.iloc[train_sample_idx]

rf_small = RandomForestRegressor(n_estimators=10, max_depth=4, random_state=42)
rf_small.fit(X_train_small, y_train_small)

y_pred_reduced = rf_small.predict(X_test_small)
rmse_reduced = np.sqrt(mean_squared_error(y_test_small, y_pred_reduced))
print("RMSE with Reduced Training Set:", rmse_reduced)


RMSE with Reduced Training Set: 0.3046716802693504


Robustness Tests – Simplified Version

We demonstrate the robustness of the Random Forest regression model
using small samples for quick execution. The tests include:

1. Gaussian noise added to numeric features
2. Outlier injection
3. Reduced training set
